# Stufe A — Bigramm-Feuerkarte des Transkodier-Experten L33/E228

**Frage:** Wie ist eine Folge von genau **zwei Buchstaben** im Morse-Mechanismus repräsentiert — trägt jeder Buchstabe für sich (additiv), oder gibt es Paar-Einheiten? Und: feuert der Experte während des **ersten** oder des **zweiten** Buchstabens?

**Aufbau:** alle 26×26 geordneten Paare des lateinischen Alphabets, je `N_JE` Ziehungen mit dem im Pilotlauf validierten Decode-Feuerindex (keine Maskierung — reine Beobachtung). Statistik: zweistufiger **Spiegelhälften-Test** (Effekt in zwei unabhängigen Datenhälften getrennt geschätzt und über die Zellen korreliert; Null per Etikettenpermutation, im Test kalibriert) — Buchstaben überhaupt → Paare über Buchstaben hinaus. Position per Vorzeichentausch über Ziehungen.

**Voraussetzungen:** Colab-A100 (80 GB), Modell von Hugging Face. Keine weiteren Dateien. Laufzeit: Modell laden + ~30–60 min.

**`EXPERTEN` erweitern,** sobald der Voll-Scan Partner von L33/E228 liefert — der Mitschnitt erfasst ohnehin alle 256 Experten jeder eingetragenen Schicht.

Einordnung, Vorregistrierung und der Stufenplan stehen in [`LIES_MICH.md`](LIES_MICH.md).

In [ ]:
# ============================================================================
# MAPPE 04 / STUFE A - Bigramm-Feuerkarte des Transkodier-Experten L33/E228
# ============================================================================
# Braucht eine A100 (80 GB). Erwartete Laufzeit: Modell laden (einige
# Minuten) plus rund 30-60 Minuten Messung (676 Bigramme x N_JE Ziehungen).
#
# DIE FRAGE (LIES_MICH.md, Abschnitt 2): Wie ist eine Folge von genau ZWEI
# Buchstaben im Morse-Mechanismus repraesentiert - traegt jeder Buchstabe
# fuer sich (additiv), oder gibt es Paar-Einheiten, die mehr sind als die
# Summe ihrer Buchstaben? Dazu die Positionsfrage: feuert der Experte
# waehrend des ersten oder des zweiten Buchstabens?
#
# WAS DER LAUF TUT:
#   1  Fuer alle 26x26 geordneten Buchstabenpaare des lateinischen
#      Alphabets je N_JE Ziehungen: "schreibe 'xy' in Morse" - in N_JE
#      RUNDEN, sodass die Ziehungen einer Zelle aus verschiedenen
#      generate-Aufrufen mit verschiedener Stapel-Nachbarschaft stammen
#      (sonst saessen beide Spiegelhaelften einer Zelle im selben Aufruf
#      und teilten dessen Stoerungen - Review-Befund). Waehrenddessen
#      laeuft der DECODE-FEUERINDEX aus dem Pilotlauf mit (Routerlogits und
#      top-8 je Antwortposition, alle 256 Experten der Zielschichten).
#   2  Antworten werden in ihre zwei Morse-Gruppen zerlegt und jedes
#      Antwort-Token dem ersten Buchstaben, dem zweiten oder keinem
#      zugeordnet (Praefix-Dekodierung, zeichengenau).
#   3  Statistik in zwei Stufen, beide als SPIEGELHAELFTEN-TEST: der
#      fragliche Effekt wird in zwei unabhaengigen Datenhaelften getrennt
#      geschaetzt und ueber die Zellen korreliert, die Null permutiert
#      Buchstaben-Etiketten (exakt austauschbar, kalibriert im Test):
#        Stufe 1  haengt das Feuern ueberhaupt an den Buchstaben?
#        Stufe 2  gibt es Paarstruktur UEBER die Buchstaben hinaus?
#      Fuer die VORREGISTRIERTEN Zellen zusaetzlich der gerichtete
#      Kandidatentest (Rang des Zellrests unter allen Zellen - dort hat
#      der globale Test allein zu wenig Power). Dazu der
#      Positionsvergleich (Vorzeichentausch ueber ZIEHUNGEN).
#
# URTEILE: MESSFELD-TOT, MESSFELD-LUECKIG, RAHMEN-ZU-DUENN,
#          BUCHSTABEN-BLIND, BUCHSTABEN-ADDITIV, PAARE-EIGEN;
#          Position: ERSTBUCHSTABE-LASTIG / ZWEITBUCHSTABE-LASTIG /
#          GLEICHVERTEILT / POSITION-UNGEMESSEN.
#
# EXPERTEN ist eine LISTE von (Schicht, Experte): sobald der laufende
# Voll-Scan Partner von E228 liefert, hier eintragen - der Mitschrieb
# erfasst ohnehin alle 256 Experten jeder eingetragenen Schicht, die
# Auswertung laeuft je Eintrag. Es wird NICHTS maskiert: dieser Lauf ist
# reine Beobachtung; Kausalitaet an Kandidatenzellen ist Stufe C.
#
# Keine weiteren Daten noetig (kein weird_transcripts.jsonl) - nur das
# Modell von Hugging Face. Ergebnisse (JSON, npz, Protokoll) schreibt die
# Zelle nach Drive in einen eigenen Lauf-Ordner; den Ordner bitte KOMPLETT
# zurueckgeben.
#
# Die Haken sind reine Mitleser (geben None zurueck, aendern nichts am
# Lauf). Die Wiederherstellungspruefung am Ende ist STRUKTURELL - sie
# zaehlt verbliebene Haken an den Modulen, statt Logits zweier
# verschiedener Texte zu vergleichen (die Lehre aus dem Fehlalarm des
# Pilotlaufs vom 2026-08-08).
#
# Die gesamte Logik zwischen den Marken "reine Logik" ist offline gegen
# Miniaturwelten mit gepflanzter Wahrheit gelaufen:
# tests/test_bigramm_logik.py, alle Tests gruen vor dem Einchecken.
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF","expandable_segments:True")
import re, math, collections, unicodedata, random, contextlib
import numpy as np, glob, json, gc, sys, time
import torch
gc.collect(); torch.cuda.empty_cache()
try: torch.cuda.synchronize()
except Exception: pass
_free=torch.cuda.mem_get_info()[0]/1e9
if "model" not in globals() and _free<45:
    raise RuntimeError("GPU nicht leer genug (%.1f GB frei, ~45 noetig)."%_free)
if not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive; drive.mount("/content/drive")
# ---------------- Protokoll und Ergebnisse automatisch nach Drive -----------
WC_RUN=globals().get("WC_RUN","bigramm_stufeA_l33_e228")
RUN_OUT="/content/drive/MyDrive/WeirdChat_Runs/%s_%s"%(WC_RUN,time.strftime("%Y%m%d-%H%M%S"))
os.makedirs(RUN_OUT,exist_ok=True)
class _WCTee:
    """Schreibt alles doppelt: in die Zelle und nach Drive. Zusaetzlich ein
       Speicherpuffer, den wc_save_all am Ende in EINEM geschlossenen
       Schreibvorgang ablegt - eine offen gehaltene Datei taucht auf dem
       Drive-Einhang nicht zuverlaessig auf."""
    _wc_tee=True
    def __init__(self,p,o): self.o=o; self.f=None; self.puffer=[]; self.retarget(p)
    def retarget(self,p):
        try:
            if self.f: self.f.close()
        except Exception: pass
        self.pfad=p; self.puffer=[]
        try: self.f=open(p,"a",encoding="utf-8")
        except Exception: self.f=None
    def write(self,s):
        self.o.write(s)
        try: self.puffer.append(s)
        except Exception: pass
        if self.f:
            try: self.f.write(s); self.f.flush()
            except Exception: pass
        return len(s)
    def flush(self):
        self.o.flush()
        if self.f:
            try: self.f.flush()
            except Exception: pass
    def isatty(self): return False
_wc_log=os.path.join(RUN_OUT,"protokoll.txt")
if getattr(sys.stdout,"_wc_tee",False): sys.stdout.retarget(_wc_log)
else: sys.stdout=_WCTee(_wc_log,sys.stdout)
def wc_save(name,obj):
    def _e(o):
        if isinstance(o,np.ndarray): return o.tolist()
        if isinstance(o,(np.integer,)): return int(o)
        if isinstance(o,(np.floating,)): return float(o)
        if isinstance(o,(np.bool_,)): return bool(o)
        return str(o)
    try:
        with open(os.path.join(RUN_OUT,name+".json"),"w",encoding="utf-8") as f:
            json.dump(obj,f,ensure_ascii=False,indent=1,default=_e)
        print("gespeichert: %s.json"%name)
    except Exception as _ex: print("konnte %s nicht speichern: %s"%(name,_ex))
def wc_protokoll_ablegen():
    try:
        _t=sys.stdout
        if getattr(_t,"_wc_tee",False) and getattr(_t,"puffer",None) is not None:
            _p=os.path.join(RUN_OUT,"protokoll_kopie.txt")
            with open(_p,"w",encoding="utf-8") as _f: _f.write("".join(_t.puffer))
            return _p
    except Exception as _ex:
        print("Protokollkopie fehlgeschlagen: %s"%_ex)
    return None
def wc_save_all():
    for _k in [k for k in list(globals()) if k.endswith("_RESULTS")]:
        wc_save(_k,globals()[_k])
    _p=wc_protokoll_ablegen()
    if _p: print("Protokollkopie:",os.path.basename(_p))
    print("Lauf-Ordner:",RUN_OUT)
print("Lauf-Ordner (Protokoll + Ergebnisse):",RUN_OUT)
if "model" not in globals() or "tokenizer" not in globals():
    from transformers import AutoModelForCausalLM, AutoTokenizer
    MODEL_ID=globals().get("MODEL_ID","Qwen/Qwen3.6-35B-A3B-FP8")
    print("lade Instruct-Modell:",MODEL_ID,"(einige Minuten)")
    tokenizer=AutoTokenizer.from_pretrained(MODEL_ID)
    model=AutoModelForCausalLM.from_pretrained(MODEL_ID,device_map="auto",torch_dtype="auto")
    model.eval()
    print("geladen | dtype:",next(model.parameters()).dtype)
# ---------------- Parameter -------------------------------------------------
# MAPPE_SEED: eigener Namensraum "bigramm/" - teilt keine Ziehung mit den
# Referenzlaeufen (20260814) oder dem Pilot (260808).
# WIEDERHOLUNG: Durchgangszaehler. DERSELBE Wert reproduziert die
# Ziehungssteuerung deterministisch; bitgleiche Antworten gibt das nur auf
# derselben Maschine mit denselben Versionen und deterministischen Kerneln
# (FP8-/MoE-Kernel garantieren das nicht, und bei TEMP=1.0 laesst ein
# gekipptes Logit-Bit die ganze Fortsetzung divergieren). Wer unabhaengig
# wiederholen will, zaehlt HOCH (Lehre aus Phase 18, Abschnitt 0c).
MAPPE_SEED=int(globals().get("MAPPE_SEED",20260815))
assert MAPPE_SEED not in (20260814,260808), \
    "MAPPE_SEED %d gehoert den Referenz-/Pilotlaeufen - eigenen Wert setzen"%MAPPE_SEED
WIEDERHOLUNG=int(globals().get("WIEDERHOLUNG",0))
EXPERTEN=list(globals().get("EXPERTEN",[(33,228)]))
ALPHABET_NAME=str(globals().get("ALPHABET_NAME","lat26"))
N_JE=int(globals().get("N_JE",12))
STAPEL=int(globals().get("STAPEL",48))
N_TEILUNGEN=int(globals().get("N_TEILUNGEN",5))
MAX_NEW=int(globals().get("MAX_NEW",24))
TEMP=float(globals().get("TEMP",1.0))
N_PERM_Z=int(globals().get("N_PERM_Z",2000))
# MIN_GUELTIG 0.6: der Ratenwuerfel nimmt ohnehin nur gueltige Antworten,
# aber wenn mehr als 40 % des Materials Prosa/Echo/Abbruch sind, ist der
# RAHMEN das Problem und kein Urteil ueber Buchstaben zu trauen.
# MIN_ZELLEN 0.7: unterhalb davon frisst die NaN-Maske der fehlenden
# Zellen die Kalibrierung des Spiegeltests an.
MIN_GUELTIG=float(globals().get("MIN_GUELTIG",0.6))
MIN_ZELLEN=float(globals().get("MIN_ZELLEN",0.7))
MIN_FEUER=float(globals().get("MIN_FEUER",0.05))
FRAME=str(globals().get("FRAME",
    'Write the two-letter sequence "%s" in International Morse code. '
    'Answer with only the Morse code, one group per letter, '
    'groups separated by a single space.'))
def prompt_text(u):
    return "<|im_start|>user\n"+u+"<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n"
def saat(zweck,schl):
    """FNV-1a wie in phase18_kern/Pilot, aber im eigenen Namensraum
       bigramm/<WIEDERHOLUNG>/ am MAPPE_SEED aufgehaengt."""
    h=2166136261
    for c in ("bigramm/%d/%s/%s"%(WIEDERHOLUNG,zweck,schl)).encode():
        h=((h^c)*16777619)&0xFFFFFFFF
    return (MAPPE_SEED+h)%(2**31-1)
def _sync():
    try: torch.cuda.synchronize()
    except Exception: pass
# ---------------- 0  Architektur --------------------------------------------
print(""); print("="*80); print("0  ARCHITEKTUR"); print("="*80)
cfg=model.config
RX=re.compile(r"^(?:model\.)?(?:language_model\.)?(?:model\.)?layers\.(\d+)\.mlp\.experts$")
RXG=re.compile(r"^(?:model\.)?(?:language_model\.)?(?:model\.)?layers\.(\d+)\.mlp\.gate$")
EXPM={}; GATEM={}
for nm,mod in model.named_modules():
    m=RX.match(nm)
    if m: EXPM[int(m.group(1))]=mod
    m=RXG.match(nm)
    if m: GATEM[int(m.group(1))]=mod
SCHICHTEN=sorted({s for s,_ in EXPERTEN})
ARCH_OK=bool(EXPM) and bool(GATEM)
if ARCH_OK:
    # In try/except: bei einer abweichenden transformers-Integration
    # (ModuleList je Experte, verschachtelte Config) soll das VERDIKT
    # fallen, nicht ein AttributeError-Traceback (Review-Befund).
    try:
        e0=EXPM[min(EXPM)]
        GU=e0.gate_up_proj; INTER=int(e0.intermediate_dim); NEXP=int(GU.shape[0])
        TOPK=int(cfg.num_experts_per_tok)
        ARCH_OK=(GU.ndim==3 and int(GU.shape[1])==2*INTER and int(GU.shape[2])==int(cfg.hidden_size)
                 and all(s in EXPM and s in GATEM for s in SCHICHTEN)
                 and all(e<NEXP for _,e in EXPERTEN))
    except Exception as _ex:
        print("  Architekturzugriff fehlgeschlagen:",_ex); ARCH_OK=False
    print("  %d Schichten | %d Experten je Schicht | top-%d | Mitschnitt an L%s"
          %(len(EXPM),NEXP,TOPK,",".join("%d"%s for s in SCHICHTEN)))
    print("  Formen wie erwartet: %s"%("ja" if ARCH_OK else "NEIN"))
if not ARCH_OK:
    BIGRAMM_RESULTS=dict(verdict="ARCHITEKTUR-NICHT-GEFUNDEN",arch_ok=False)
    wc_save_all(); print(""); print("VERDIKT: ARCHITEKTUR-NICHT-GEFUNDEN"); raise SystemExit(0)
try: DTYPE=str(next(model.parameters()).dtype)
except Exception: DTYPE="unbekannt"
EOS_ID=int(tokenizer.eos_token_id)
PAD_ID=int(tokenizer.pad_token_id) if tokenizer.pad_token_id is not None else EOS_ID
# ALLE Stopp-IDs des Modells: Qwen-Instruct stoppt laut generation_config
# auch auf endoftext, nicht nur auf im_end. Eine nur an im_end geschnittene
# Reihe liesse die Fuelltoken-Schritte nach dem Stopp mitzaehlen.
_ge=getattr(getattr(model,"generation_config",None),"eos_token_id",None)
STOPP_IDS=sorted({EOS_ID,PAD_ID}
                 |({int(x) for x in _ge} if isinstance(_ge,(list,tuple))
                   else ({int(_ge)} if _ge is not None else set())))
STOPP=frozenset(STOPP_IDS)
print("  Stopp-IDs:",STOPP_IDS)
# ---------------- Werkzeuge (Apparat wie im Pilot, mehrschichtfaehig) -------
tokenizer.padding_side="left"
assert tokenizer.padding_side=="left", \
    "links auffuellen ist Pflicht: rechts eingefuegte Fuelltoken staenden zwischen Prompt und Antwort"
class FeuerIndexM:
    """Decode-Feuerindex des Pilotlaufs, verallgemeinert auf mehrere
       Schichten: je Zielschicht die Routerlogits (gate-forward) und die
       top-8-Auswahl (experts-pre) jeder Decode-Position. Der Filter auf
       fl.shape[0]==stapel laesst nur Decode-Schritte durch (Prefill hat
       stapel*Promptlaenge Zeilen). Die Haken LESEN nur (geben None
       zurueck) - am Lauf aendert sich nichts."""
    def __init__(self,stapel,schichten):
        self.stapel=int(stapel); self.schichten=list(schichten)
        self.logits={s:[] for s in self.schichten}
        self.top8={s:[] for s in self.schichten}
        self.griffe=[]
    def __enter__(self):
        def _gate(s):
            def h(mod,inp,out):
                lg=out[0] if isinstance(out,(tuple,list)) else out
                fl=lg.detach().reshape(-1,lg.shape[-1])
                if int(fl.shape[0])==self.stapel:
                    self.logits[s].append(np.asarray(fl.float().cpu(),dtype=np.float32))
                return None
            return h
        def _pre(s):
            def h(mod,args):
                idx=args[1].detach()
                fl=idx.reshape(-1,idx.shape[-1])
                if int(fl.shape[0])==self.stapel:
                    self.top8[s].append(np.asarray(fl.cpu()).astype(np.int32))
                return None
            return h
        for s in self.schichten:
            self.griffe+=[GATEM[s].register_forward_hook(_gate(s)),
                          EXPM[s].register_forward_pre_hook(_pre(s))]
        return self
    def __exit__(self,*a):
        for g in self.griffe: g.remove()
        self.griffe=[]
        return False
    def stapel_arrays(self,s):
        assert len(self.logits[s])==len(self.top8[s]), \
            "gate- und experts-Mitschnitt ungleich lang (L%d: %d/%d)"%(s,len(self.logits[s]),len(self.top8[s]))
        if not self.logits[s]: return None
        return (np.stack(self.logits[s],axis=1),np.stack(self.top8[s],axis=1))
def zieh_gemischt(texte,startwert):
    """EIN generate-Aufruf fuer VERSCHIEDENE Prompts (das ist neu gegenueber
       dem Pilot, der je Aufruf einen Text vervielfaeltigte - darum oben das
       Links-Auffuellen). Rueckgabe: Antwort-IDs (n, MAX_NEW) und je
       Zielschicht die Mitschnitt-Arrays."""
    enc=tokenizer(texte,return_tensors="pt",padding=True).to(model.device)
    L=int(enc["input_ids"].shape[1])
    torch.manual_seed(startwert)
    with FeuerIndexM(len(texte),SCHICHTEN) as rec:
        with torch.no_grad():
            g=model.generate(**enc,do_sample=True,temperature=TEMP,top_p=1.0,top_k=0,
                             repetition_penalty=1.0,max_new_tokens=MAX_NEW,
                             pad_token_id=PAD_ID)
    ids=np.asarray(g.cpu())[:,L:].astype(np.int32)
    mit={s:rec.stapel_arrays(s) for s in SCHICHTEN}
    return ids,mit
def antwort_texte(ids_reihe):
    """An ALLEN Stopp-IDs beschnittene Antwort einer Reihe: (Volltext,
       Praefixstuecke). Beide aus demselben Dekodierweg, damit die
       Zeichenspannen zum Volltext passen. Der Schnitt am ersten Stopp-
       Token (im_end ODER endoftext) haelt auch die Fuelltoken heraus,
       die generate nach dem Ende einer Reihe nachschiebt."""
    t=[int(x) for x in ids_reihe]
    schnitt=[i for i,x in enumerate(t) if x in STOPP]
    if schnitt: t=t[:schnitt[0]]
    stuecke=[tokenizer.decode(t[:k+1],skip_special_tokens=True) for k in range(len(t))]
    return (stuecke[-1] if stuecke else ""),stuecke
def _pad_zu(a,S,wert):
    if a.shape[1]==S: return a
    form=(a.shape[0],S-a.shape[1])+a.shape[2:]
    return np.concatenate([a,np.full(form,wert,dtype=a.dtype)],axis=1)
def fuege_zusammen(fi,pad_id):
    """Bloecke verschiedener Laenge auf eine Laenge bringen: Logits mit NaN,
       top-8 mit -1, Token mit pad_id (unveraendert aus dem Pilot)."""
    fi=[f for f in fi if f["top8"] is not None]
    if not fi: return None
    S=max(f["top8"].shape[1] for f in fi); Tn=max(f["tokens"].shape[1] for f in fi)
    lo=np.concatenate([_pad_zu(f["logits"].astype(np.float16),S,np.float16(np.nan)) for f in fi])
    t8=np.concatenate([_pad_zu(f["top8"].astype(np.int16),S,np.int16(-1)) for f in fi])
    tk=np.concatenate([_pad_zu(f["tokens"],Tn,np.int32(pad_id)) for f in fi])
    return dict(logits=lo,top8=t8,tokens=tk)
# ---- reine Logik: ANFANG ----------------------------------------------------
# Alles zwischen ANFANG und ENDE ist reines numpy, laeuft ohne GPU und wird
# von tests/test_bigramm_logik.py gegen Miniaturwelten mit gepflanzter
# Wahrheit geprueft. Funktionen nehmen ihre Parameter als Argumente - keine
# versteckten Globalen ausser numpy als np.
ALPHABETE={"lat26":"abcdefghijklmnopqrstuvwxyz"}
MORSE_INT={"a":".-","b":"-...","c":"-.-.","d":"-..","e":".","f":"..-.",
           "g":"--.","h":"....","i":"..","j":".---","k":"-.-","l":".-..",
           "m":"--","n":"-.","o":"---","p":".--.","q":"--.-","r":".-.",
           "s":"...","t":"-","u":"..-","v":"...-","w":".--","x":"-..-",
           "y":"-.--","z":"--.."}
# Vorregistrierte Kandidatenzellen (LIES_MICH, Abschnitt "Vorregistrierung"):
# "ch" ist im deutschen Landes-Morse ein EIGENES Zeichen (----) - wenn es
# Paar-Einheiten gibt, ist das die erste Adresse. Der Rest sind haeufige
# deutsche Digraphen, erklaertermassen nachrangig.
KANDIDATEN_PRIMAER=[("c","h")]
KANDIDATEN_SEKUNDAER=[("e","n"),("e","r"),("e","i"),("s","t"),("c","k")]
def bigramme(buchstaben):
    """Alle geordneten Paare, Reihenfolge deterministisch (a-vor-b-Schleife)."""
    return [(a,b) for a in buchstaben for b in buchstaben]
def normalisiere_morse(text):
    """Zeichenvarianten auf ./- ziehen: Mittelpunkt/Bullet auf Punkt,
       Minus/Gedankenstriche/Unterstrich auf Strich."""
    aus=[]
    for c in text:
        if c in "·•∙.": aus.append(".")
        elif c in "-−–—_‐‑": aus.append("-")
        else: aus.append(c)
    return "".join(aus)
def zerlege_antwort(text):
    """Morse-Gruppen der Antwort mit Zeichenspannen: Laeufe aus Punkt/Strich-
       Varianten, getrennt durch beliebige andere Zeichen. Rueckgabe
       [(gruppe_normalisiert, anfang, ende_exklusiv), ...]."""
    norm=normalisiere_morse(text)
    gruppen=[]; i=0
    while i<len(norm):
        if norm[i] in ".-":
            j=i
            while j<len(norm) and norm[j] in ".-": j+=1
            gruppen.append((norm[i:j],i,j)); i=j
        else: i+=1
    return gruppen
def gueltige_zwei(gruppen):
    """Gueltig ist eine Antwort mit GENAU zwei nichtleeren Morse-Gruppen -
       eine je Buchstabe. Mehr oder weniger heisst: die Ziehung ist nicht
       lesbar (Echo des Prompts, Prosa, Abbruch)."""
    return len(gruppen)==2 and all(g[0] for g in gruppen)
def antwort_korrekt(gruppen,a,b,tabelle):
    if not gueltige_zwei(gruppen): return False
    return gruppen[0][0]==tabelle.get(a) and gruppen[1][0]==tabelle.get(b)
def token_spannen(stuecke):
    """Zeichenspannen je Antwort-Token aus den dekodierten PRAEFIXTEXTEN
       (Eintrag j = Text nach Token j). Byte-BPE kann Praefixlaengen kurz
       schrumpfen lassen (halbe UTF-8-Zeichen als Ersatzzeichen) - Spannen
       werden darum geklemmt, nie negativ."""
    spannen=[]; vor=0
    for s in stuecke:
        ende=len(s); anfang=min(vor,ende)
        spannen.append((anfang,ende)); vor=ende
    return spannen
def token_klassen(spannen,gruppen):
    """Klasse je Token: 0 = ueberlappt nur Gruppe 1, 1 = nur Gruppe 2,
       -1 = leer, keine oder beide (Trenner, Vortext, Grenzueberspanner)."""
    if not gueltige_zwei(gruppen): return [-1]*len(spannen)
    (_,a1,e1),(_,a2,e2)=gruppen
    kl=[]
    for (s,e) in spannen:
        if e<=s: kl.append(-1); continue
        in1=(s<e1 and e>a1); in2=(s<e2 and e>a2)
        if in1 and not in2: kl.append(0)
        elif in2 and not in1: kl.append(1)
        else: kl.append(-1)
    return kl
def feuer_je_klasse(top8,tokens,klassenliste,eos_id,ziel_e):
    """Wie feuer_auswertung im Pilot, aber je Tokenklasse getrennt gezaehlt.
       Rueckgabe je Ziehung [[treffer,gueltig] fuer Klasse 0, 1, -1];
       Mitschnitt s gehoert zur Antwortposition s. eos_id darf eine MENGE
       von Stopp-IDs sein: Qwen-Instruct stoppt laut generation_config auch
       auf endoftext (nicht nur im_end), und nach dem Stopp einer Reihe
       schiebt generate Fuelltoken nach, solange andere Reihen laufen -
       deren Router-Schritte duerfen nicht mitzaehlen (Review-Befund)."""
    stopp=set(int(x) for x in eos_id) if isinstance(eos_id,(set,frozenset,list,tuple))           else {int(eos_id)}
    n,S=top8.shape[0],top8.shape[1]
    je=[]
    for i in range(n):
        t=np.asarray(tokens[i])
        stop=np.where(np.isin(t,list(stopp)))[0]
        ende=int(stop[0]) if stop.size else int(t.shape[0])
        gueltig=min(ende,S,len(klassenliste[i]))
        z={0:[0,0],1:[0,0],-1:[0,0]}
        for s in range(gueltig):
            if int(top8[i,s,0])<0: continue
            k=klassenliste[i][s]
            z[k][1]+=1
            z[k][0]+=int(ziel_e in set(int(x) for x in top8[i,s]))
        je.append([z[0],z[1],z[-1]])
    return je
def zaehlwerk_eintrag(zaehl,schluessel,je_reihe,gueltig):
    """Traegt eine Reihe ins Zaehlwerk fuer den Ratenwuerfel ein - NUR wenn
       die Antwort gueltig ist. Ungueltige Antworten (Prosa, Echo, Abbruch)
       entstehen selbst buchstabenabhaengig; ihr Feuern wuerde als
       replizierbare Struktur in R einsickern und waere vom Transkodier-
       Feuern nicht zu unterscheiden (Review-Befund). Rueckgabe (tr,g) der
       Reihe in jedem Fall - fuer die Roh-Gesamtrate ueber ALLE Reihen."""
    tr=sum(x[0] for x in je_reihe); g=sum(x[1] for x in je_reihe)
    if gueltig and g>0: zaehl[schluessel]=[tr,g]
    return tr,g
def raten_wuerfel(zaehl,A,B,n_je):
    """Aus dem Zaehlwerk (dict (ai,bi,k) -> [treffer,gueltig]) den Ratenwuerfel
       R[ai,bi,k] bauen; Ziehungen ohne gueltige Positionen bleiben NaN."""
    R=np.full((A,B,n_je),np.nan)
    for (ai,bi,k),(tr,g) in zaehl.items():
        if g>0: R[ai,bi,k]=tr/g
    return R
def additiv_parameter(M,runden=50):
    """Kleinste Quadrate fuer M[a,b] ~ mu + alpha_a + beta_b mit Luecken
       (NaN), per abwechselndem Zeilen/Spalten-Abgleich. Rueckgabe
       (mu, alpha_intern, beta_intern, alpha, beta); die internen Vektoren
       sind 0-gefuellt (fuer die Matrixrekonstruktion), die aeusseren tragen
       NaN, wo eine Zeile/Spalte gar keine Daten hat - eine erfundene 0
       wuerde sonst in beiden Haelften gleich aussehen und Korrelation
       vortaeuschen."""
    mu=float(np.nanmean(M))
    alpha=np.zeros(M.shape[0]); beta=np.zeros(M.shape[1])
    with np.errstate(invalid="ignore"):
        for _ in range(runden):
            alpha=np.nan_to_num(np.nanmean(M-mu-beta[None,:],axis=1))
            beta=np.nan_to_num(np.nanmean(M-mu-alpha[:,None],axis=0))
            mu=mu+float(np.nanmean(M-mu-alpha[:,None]-beta[None,:]))
    a_aus=np.where(np.any(np.isfinite(M),axis=1),alpha,np.nan)
    b_aus=np.where(np.any(np.isfinite(M),axis=0),beta,np.nan)
    return mu,alpha,beta,a_aus,b_aus
def passe_additiv(M,runden=50):
    mu,al,be,_,_=additiv_parameter(M,runden)
    return mu+al[:,None]+be[None,:]
def _korr(x,y):
    """Pearson ueber gemeinsam endliche Eintraege; NaN wenn zu wenige."""
    ok=np.isfinite(x)&np.isfinite(y)
    if int(np.sum(ok))<3: return np.nan
    x=x[ok]-np.mean(x[ok]); y=y[ok]-np.mean(y[ok])
    nx=float(np.sqrt(np.sum(x*x))); ny=float(np.sqrt(np.sum(y*y)))
    if nx==0.0 or ny==0.0: return np.nan
    return float(np.sum(x*y)/(nx*ny))
def teile_haelften(n_je,rnd):
    idx=list(range(n_je)); rnd.shuffle(idx)
    h=n_je//2
    return idx[:h],idx[h:]
def _haelften_mittel(R,ia,ib):
    with np.errstate(invalid="ignore"):
        return np.nanmean(R[:,:,ia],axis=2),np.nanmean(R[:,:,ib],axis=2)
def _spiegel_schaetzer(R,stufe,rnd,n_teilungen):
    """Je Haelften-Teilung die beiden unabhaengigen Schaetzer des fraglichen
       Effekts (Stufe 1: Zeilen-/Spalteneffekte, Stufe 2: Interaktionsreste)."""
    A,B,n_je=R.shape
    aus=[]
    for _ in range(n_teilungen):
        ia,ib=teile_haelften(n_je,rnd)
        RA,RB=_haelften_mittel(R,ia,ib)
        if stufe==1:
            _,_,_,aA,bA=additiv_parameter(RA)
            _,_,_,aB,bB=additiv_parameter(RB)
            aus.append((np.concatenate([aA,bA]),(aB,bB)))
        else:
            aus.append((RA-passe_additiv(RA),RB-passe_additiv(RB)))
    return aus
def p_spiegel(R,stufe,n_perm,rnd,n_teilungen=5):
    """Spiegelhaelften-Test. Die Ziehungen jeder Zelle werden in zwei
       Haelften geteilt, der fragliche Effekt in JEDER Haelfte unabhaengig
       geschaetzt und ueber die Zellen korreliert - echte Struktur
       repliziert sich, Rauschen nicht. Statistik ist das MITTEL der
       Spiegelkorrelation ueber n_teilungen zufaellige Teilungen: an einem
       einzelnen Schnitt hinge das Urteil an der Saat (Review-Befund).
       Die Null permutiert ETIKETTEN der zweiten Haelfte (Stufe 1:
       Buchstabenetiketten der Effektvektoren; Stufe 2: Zeilen und Spalten
       der Interaktionsmatrix, was deren Summenstruktur erhaelt). Unter der
       Null sind die Etiketten austauschbar, WEIL der additive Anteil vor
       der Statistik abgezogen ist - starke Buchstabeneffekte koennen
       Stufe 2 darum nicht verfaelschen. Die Korrelation ist
       selbstzentrierend: ihr Nullmittel haengt nicht vom Rauschpegel ab
       (der Fehler des zuvor erwogenen Aus-der-Stichprobe-Gewinns, 13.5 %
       Fehlalarm in der Kalibrierung). Einseitig: echte Struktur
       korreliert positiv. Ziehungen bleiben als unabhaengige Einheiten
       beisammen - Hausregel 2. Kalibrierung und Power (auch mit
       Binomialrauschen) stehen als Tests in der Suite."""
    A,B,n_je=R.shape
    schaetzer=_spiegel_schaetzer(R,stufe,rnd,n_teilungen)
    beobs=[]
    for links,rechts in schaetzer:
        if stufe==1:
            aB,bB=rechts; r=_korr(links,np.concatenate([aB,bB]))
        else:
            r=_korr(links.ravel(),rechts.ravel())
        if np.isfinite(r): beobs.append(r)
    if not beobs: return 1.0,np.nan
    beob=float(np.mean(beobs))
    t=0
    for _ in range(n_perm):
        # EINE Etikettenpermutation je Nullziehung, auf ALLE Teilungen
        # angewandt: die beobachteten Teilungs-Korrelationen teilen sich
        # Daten und sind untereinander korreliert - frische Permutationen
        # je Teilung wuerden diese Struktur zerstoeren und die Null zu eng
        # machen (in der Kalibrierung gemessen: 20 % Fehlalarm). Eine
        # gemeinsame Umbenennung der Zellen ist unter der Null
        # masserhaltend und laesst die Kreuz-Teilungs-Abhaengigkeit stehen.
        pa=list(range(A)); rnd.shuffle(pa)
        pb=list(range(B)); rnd.shuffle(pb)
        vals=[]
        for links,rechts in schaetzer:
            if stufe==1:
                aB,bB=rechts
                r=_korr(links,np.concatenate([aB[pa],bB[pb]]))
            else:
                r=_korr(links.ravel(),rechts[np.ix_(pa,pb)].ravel())
            if np.isfinite(r): vals.append(r)
        if vals and float(np.mean(vals))>=beob-1e-12: t+=1
    return (t+1)/(n_perm+1.0),beob
def p_kandidat(R,zi,zj):
    """Gerichteter Test fuer EINE vorregistrierte Zelle: Rang ihres
       Interaktionsrests unter allen Zellen (einseitig). Unter der
       additiven Null sind die Zellreste austauschbar - der Rang ist ein
       exakter p-Wert. Fuer eine einzelne Zelle hat dieser Test die Power,
       die der globale Spiegeltest dort nicht hat (Review-Befund: eine
       Zelle unter 676 verschwindet in der Nullstreuung ~1/sqrt(676))."""
    M=np.nanmean(R,axis=2)
    g=M-passe_additiv(M)
    x=g[zi,zj]
    alle=g[np.isfinite(g)]
    if not np.isfinite(x) or alle.size<10: return 1.0
    return float(np.sum(alle>=x-1e-12)/alle.size)
def paar_reste(R):
    """Interaktionsschaetzer je Zelle: Zellmittel minus additive Anpassung."""
    M=np.nanmean(R,axis=2)
    return M-passe_additiv(M)
def spitzen_zellen(R,buchstaben,hoechstens=10,mindest_ziehungen=3):
    """Betragsstaerkste Interaktionszellen, stabilisiert: gamma geteilt
       durch seinen Standardfehler aus den gueltigen Ziehungen der Zelle,
       und Zellen unter mindest_ziehungen fliegen raus - sonst fuehren
       die duennsten Zellen per Rauschen die Liste an und lenken
       Stufe-C-GPU-Zeit auf Artefakte (Review-Befund). Kandidaten werden
       markiert, die Belegung wird mitgegeben."""
    gamma=paar_reste(R)
    kand=set(KANDIDATEN_PRIMAER)|set(KANDIDATEN_SEKUNDAER)
    zellen=[]
    for i,a in enumerate(buchstaben):
        for j,b in enumerate(buchstaben):
            g=gamma[i,j]
            x=R[i,j,:]; x=x[np.isfinite(x)]
            if not np.isfinite(g) or x.size<mindest_ziehungen: continue
            se=float(np.std(x,ddof=1))/np.sqrt(x.size) if x.size>1 else 0.0
            z=float(g)/se if se>1e-9 else 0.0
            zellen.append((abs(z),z,float(g),int(x.size),a,b,(a,b) in kand))
    zellen.sort(key=lambda z:(-z[0],z[4],z[5]))
    return [dict(paar=a+b,z=z,gamma=g,ziehungen=m,kandidat=k)
            for _,z,g,m,a,b,k in zellen[:hoechstens]]
def positions_vergleich(je_klasse,n_perm,rnd):
    """Feuert der Experte beim ersten oder beim zweiten Buchstaben? Je Ziehung
       (unabhaengige Einheit) die Ratendifferenz Klasse0 minus Klasse1, dann
       Vorzeichentausch-Null UEBER ZIEHUNGEN, nicht ueber Token."""
    d=[]
    for z in je_klasse:
        (t0,g0),(t1,g1)=z[0],z[1]
        if g0>0 and g1>0: d.append(t0/g0-t1/g1)
    if len(d)<8: return 1.0,0.0,len(d),(float("-inf"),float("inf"))
    d=np.asarray(d); beob=float(np.mean(d)); t=0
    for _ in range(n_perm):
        vz=np.asarray([1.0 if rnd.random()<0.5 else -1.0 for _ in d])
        if abs(float(np.mean(d*vz)))>=abs(beob)-1e-12: t+=1
    sf=float(np.std(d,ddof=1))/np.sqrt(len(d))
    ki=(beob-1.96*sf,beob+1.96*sf)
    return (t+1)/(n_perm+1.0),beob,len(d),ki
def urteil_position(p,delta,n,ki,alpha=0.05,gleich_band=0.1):
    """GLEICHVERTEILT ist eine POSITIVE Behauptung - sie verlangt nicht nur
       p>=alpha, sondern ein Konfidenzintervall, das ganz im Band
       (-gleich_band,+gleich_band) liegt. Nichtablehnung bei breitem KI
       heisst nur POSITION-UNENTSCHIEDEN (Review-Befund)."""
    if n<8: return "POSITION-UNGEMESSEN"
    if p<alpha:
        return "ERSTBUCHSTABE-LASTIG" if delta>0 else "ZWEITBUCHSTABE-LASTIG"
    if ki[0]>-gleich_band and ki[1]<gleich_band: return "GLEICHVERTEILT"
    return "POSITION-UNENTSCHIEDEN"
def urteil_bigramm(gueltig_anteil,zellen_abdeckung,feuer_rate,p_buchstaben,
                   p_paare,min_gueltig=0.5,min_zellen=0.7,min_feuer=0.05,
                   alpha=0.05):
    """Reihenfolge der Tore wie immer: erst stirbt die Messung, dann kommt
       das Urteil. RAHMEN-ZU-DUENN heisst: der Minimalrahmen rekrutiert den
       Mechanismus nicht - erst den Rahmen anreichern (z. B. ins WeirdChat-
       Szenario einbetten), dann wieder messen."""
    if gueltig_anteil<min_gueltig: return "MESSFELD-TOT"
    if zellen_abdeckung<min_zellen: return "MESSFELD-LUECKIG"
    if feuer_rate is None or feuer_rate<min_feuer: return "RAHMEN-ZU-DUENN"
    if p_paare<alpha: return "PAARE-EIGEN"
    if p_buchstaben>=alpha: return "BUCHSTABEN-BLIND"
    return "BUCHSTABEN-ADDITIV"
# ---- reine Logik: ENDE ------------------------------------------------------
# ---------------- 1  Stimuli und Ziehungen ----------------------------------
print(""); print("="*80); print("1  STIMULI UND ZIEHUNGEN"); print("="*80)
BUCHSTABEN=ALPHABETE[ALPHABET_NAME]
PAARE=bigramme(BUCHSTABEN)
IDXB={c:i for i,c in enumerate(BUCHSTABEN)}
A=len(BUCHSTABEN)
# N_JE RUNDEN: Ziehung k jeder Zelle kommt aus Runde k, mit je Runde neu
# gemischter Reihenfolge. So stammen die Ziehungen einer Zelle aus N_JE
# verschiedenen generate-Aufrufen mit verschiedener Stapel-Nachbarschaft -
# Aufruf-Stoerungen (Seed, Stapelzusammensetzung, Padding, Kernel) koennen
# nicht in BEIDE Spiegelhaelften derselben Zelle laufen (Review-Befund).
AUFRUFE=[]
for k in range(N_JE):
    ordnung=list(range(len(PAARE)))
    random.Random(saat("ordnung","runde%d"%k)).shuffle(ordnung)
    for g0 in range(0,len(ordnung),STAPEL):
        AUFRUFE.append([(pi,k) for pi in ordnung[g0:g0+STAPEL]])
print("  Alphabet %s: %d Zeichen, %d Bigramme, %d Ziehungen je Bigramm"
      %(ALPHABET_NAME,A,len(PAARE),N_JE))
print("  %d generate-Aufrufe zu je hoechstens %d Reihen, %d Runden"
      %(len(AUFRUFE),STAPEL,N_JE))
print("  Experten in der Auswertung: %s"%", ".join("L%d/E%d"%(s,e) for s,e in EXPERTEN))
# Apparatprobe: ein Mini-Aufruf, dann Formpruefung des Mitschnitts. Faengt
# Signatur-Drift der Hooks (args[1] o. ae.) mit klarer Meldung ab, bevor
# GPU-Stunden laufen (Review-Befund).
_pids,_pmit=zieh_gemischt([prompt_text("Say OK.")]*2,saat("probe","apparat"))
for _s in SCHICHTEN:
    _st=_pmit[_s]
    assert _st is not None and _st[0].shape[0]==2 and _st[0].shape[2]==NEXP \
       and _st[1].shape[0]==2 and _st[1].shape[2]==TOPK, \
       "Apparatprobe: Mitschnitt an L%d hat unerwartete Form"%_s
print("  Apparatprobe bestanden (Mitschnitt-Formen wie erwartet)")
ZAEHL={se:{} for se in EXPERTEN}
FEUER_ROH={se:[0,0] for se in EXPERTEN}
JE_KLASSE={se:[] for se in EXPERTEN}
ROH={s:[] for s in SCHICHTEN}
REIHEN=[]
ANTWORTEN=[]
n_gueltig=0; n_korrekt=0; n_reihen=0
_sync(); t_start=time.time(); t_je_aufruf=[]
for ai,reihen in enumerate(AUFRUFE):
    texte=[prompt_text(FRAME%(PAARE[pi][0]+PAARE[pi][1])) for pi,_ in reihen]
    _sync(); t0=time.time()
    ids,mit=zieh_gemischt(texte,saat("zieh",str(ai)))
    _sync(); t_je_aufruf.append(time.time()-t0)
    klassen_liste=[]; gueltig_liste=[]
    for j,(pi,k) in enumerate(reihen):
        a,b=PAARE[pi]
        voll,stuecke=antwort_texte(ids[j])
        gruppen=zerlege_antwort(voll)
        gv=gueltige_zwei(gruppen)
        kor=antwort_korrekt(gruppen,a,b,MORSE_INT)
        kl=token_klassen(token_spannen(stuecke),gruppen)
        klassen_liste.append(kl); gueltig_liste.append(gv)
        n_reihen+=1; n_gueltig+=int(gv); n_korrekt+=int(kor)
        REIHEN.append((IDXB[a],IDXB[b],k,int(gv),int(kor)))
        ANTWORTEN.append(dict(paar=a+b,ziehung=k,text=voll,gueltig=bool(gv),
                              korrekt=bool(kor)))
    for (s,e) in EXPERTEN:
        st=mit[s]
        if st is None: continue
        je=feuer_je_klasse(st[1],ids,klassen_liste,STOPP,e)
        for j,(pi,k) in enumerate(reihen):
            a,b=PAARE[pi]
            tr,g=zaehlwerk_eintrag(ZAEHL[(s,e)],(IDXB[a],IDXB[b],k),je[j],
                                   gueltig_liste[j])
            FEUER_ROH[(s,e)][0]+=tr; FEUER_ROH[(s,e)][1]+=g
            if gueltig_liste[j]: JE_KLASSE[(s,e)].append(je[j])
    for s in SCHICHTEN:
        st=mit[s]
        ROH[s].append(dict(logits=None if st is None else st[0],
                           top8=None if st is None else st[1],tokens=ids))
    if (ai+1)%10==0 or ai+1==len(AUFRUFE):
        lauf=time.time()-t_start
        rest=lauf/(ai+1)*(len(AUFRUFE)-ai-1)
        print("  Aufruf %3d/%d | gueltig %4d/%4d | korrekt %4d | %5.1f min, noch ~%4.1f min"
              %(ai+1,len(AUFRUFE),n_gueltig,n_reihen,n_korrekt,lauf/60.0,rest/60.0))
_sync(); T_MESSUNG=time.time()-t_start
# Wiederherstellung: STRUKTURELL - keine Haken mehr an den Zielmodulen.
_haengen=sum(len(m._forward_hooks)+len(m._forward_pre_hooks)
             for s in SCHICHTEN for m in (GATEM[s],EXPM[s]))
print("  WIEDERHERSTELLUNG: %d Haken verblieben an den Zielmodulen"%_haengen)
assert _haengen==0,"Haken haengen noch"
# ---------------- Rohdaten je Schicht nach Drive ----------------------------
DATEIEN=[]
REIHEN_ARR=np.asarray(REIHEN,dtype=np.int16)
for s in SCHICHTEN:
    d=fuege_zusammen(ROH[s],PAD_ID)
    if d is None:
        print("  ACHTUNG: kein Mitschnitt fuer L%d"%s); continue
    pfad=os.path.join(RUN_OUT,"feuerwuerfel_L%d"%s)
    np.savez_compressed(pfad,logits=d["logits"],top8=d["top8"],tokens=d["tokens"],
                        reihen=REIHEN_ARR,buchstaben=BUCHSTABEN,
                        mappe_seed=MAPPE_SEED,wiederholung=WIEDERHOLUNG,
                        eos_id=EOS_ID,pad_id=PAD_ID,
                        stopp_ids=np.asarray(STOPP_IDS,dtype=np.int64),schicht=s)
    DATEIEN.append(os.path.basename(pfad)+".npz")
    print("  gespeichert: %s (%d Reihen)"%(DATEIEN[-1],d["top8"].shape[0]))
# ---------------- 2  Auswertung je Experte ----------------------------------
print(""); print("="*80); print("2  AUSWERTUNG"); print("="*80)
GUELTIG_ANTEIL=n_gueltig/max(n_reihen,1)
KORREKT_ANTEIL=n_korrekt/max(n_reihen,1)
print("  Antworten: %d | gueltig (genau zwei Morse-Gruppen): %.1f %% | korrekt: %.1f %%"
      %(n_reihen,100.0*GUELTIG_ANTEIL,100.0*KORREKT_ANTEIL))
ERG={}
for (s,e) in EXPERTEN:
    name="L%d/E%d"%(s,e)
    R=raten_wuerfel(ZAEHL[(s,e)],A,A,N_JE)
    abdeckung=float(np.mean(np.sum(np.isfinite(R),axis=2)>=2))
    ges=list(ZAEHL[(s,e)].values())
    g_sum=sum(g for _,g in ges)
    feuer=(sum(t for t,_ in ges)/g_sum) if g_sum else None
    tr_roh,g_roh=FEUER_ROH[(s,e)]
    feuer_roh=(tr_roh/g_roh) if g_roh else None
    p1,r1=p_spiegel(R,1,N_PERM_Z,random.Random(saat("stufe1",name)),N_TEILUNGEN)
    p2,r2=p_spiegel(R,2,N_PERM_Z,random.Random(saat("stufe2",name)),N_TEILUNGEN)
    oben=spitzen_zellen(R,BUCHSTABEN,hoechstens=10)
    kandidaten={}
    for (ka,kb) in KANDIDATEN_PRIMAER:
        kandidaten[ka+kb]=dict(p=p_kandidat(R,IDXB[ka],IDXB[kb]),rang="primaer")
    for (ka,kb) in KANDIDATEN_SEKUNDAER:
        kandidaten[ka+kb]=dict(p=p_kandidat(R,IDXB[ka],IDXB[kb]),rang="sekundaer")
    pp,pd,pn,pki=positions_vergleich(JE_KLASSE[(s,e)],N_PERM_Z,
                                     random.Random(saat("position",name)))
    u=urteil_bigramm(GUELTIG_ANTEIL,abdeckung,feuer,p1,p2,
                     MIN_GUELTIG,MIN_ZELLEN,MIN_FEUER)
    up=urteil_position(pp,pd,pn,pki)
    ERG[name]=dict(schicht=s,experte=e,
                   feuer_rate_gueltig=feuer,feuer_rate_roh=feuer_roh,
                   zellen_abdeckung=abdeckung,
                   p_buchstaben=p1,spiegel_r_buchstaben=r1,
                   p_paare=p2,spiegel_r_paare=r2,
                   kandidaten=kandidaten,
                   position_p=pp,position_delta=pd,position_n=pn,
                   position_ki=list(pki),
                   urteil=u,urteil_position=up,
                   raten_matrix=np.nanmean(R,axis=2),gamma_matrix=paar_reste(R),
                   spitzen=oben)
    print("")
    print("  %s  Feuerrate %s (gueltige Antworten; roh %s) | Zellen mit >=2 Ziehungen: %.1f %%"
          %(name,"-" if feuer is None else "%.3f"%feuer,
            "-" if feuer_roh is None else "%.3f"%feuer_roh,100.0*abdeckung))
    print("         Stufe 1 (Buchstaben ueberhaupt): Spiegel-r %+.3f, p=%.4f"%(r1,p1))
    print("         Stufe 2 (Paare ueber Buchstaben): Spiegel-r %+.3f, p=%.4f"%(r2,p2))
    print("         Kandidatentest (Rang unter allen Zellen): %s"
          %", ".join("%s p=%.4f (%s)"%(kz,kv["p"],kv["rang"])
                     for kz,kv in sorted(kandidaten.items())))
    print("         Position: Delta(erst-zweit)=%+.3f [%+.3f,%+.3f], p=%.4f, n=%d -> %s"
          %(pd,pki[0],pki[1],pp,pn,up))
    print("         URTEIL: %s"%u)
    print("         Spitzenzellen (|z|, mind. 3 Ziehungen): %s"
          %", ".join("%s z%+.1f (g%+.3f, %dz)%s"%(z["paar"],z["z"],z["gamma"],
                     z["ziehungen"]," *K*" if z["kandidat"] else "")
                     for z in oben[:6]))
print("")
print("  Lesehilfe: URTEIL gilt je Experte. PAARE-EIGEN heisst: die")
print("  Interaktionsreste replizieren sich zwischen unabhaengigen")
print("  Datenhaelften - es gibt Paar-Einheiten. BUCHSTABEN-ADDITIV heisst: zwei Buchstaben sind im")
print("  Feuern dieses Experten die Summe ihrer Einzelbuchstaben - dann")
print("  skaliert Stufe B mit |Alphabet|, nicht mit |Alphabet|^2.")
print("  *K* markiert vorregistrierte Kandidaten (ch zuerst - im deutschen")
print("  Landes-Morse ist ch ein EIGENES Zeichen ----).")
# ---------------- 3  Durchsatz und Hochrechnung Stufe B ---------------------
print(""); print("="*80); print("3  DURCHSATZ (bitte zurueckmelden)"); print("="*80)
s_aufruf=float(np.mean(t_je_aufruf)) if t_je_aufruf else float("nan")
SCHRIFTEN_PLAN={"lat26":26,"kyrillisch":32,"griechisch":24,"hebraeisch":22,
                "arabisch":28,"kana_wabun":46,"hangul_jamo":24}
DURCHSATZ=dict(s_je_aufruf=s_aufruf,reihen_je_aufruf=STAPEL,
               s_messung_gesamt=T_MESSUNG,hochrechnung_stufeB={})
print("  je Aufruf (%d Reihen): %.2f s | Messung gesamt: %.1f min"
      %(STAPEL,s_aufruf,T_MESSUNG/60.0))
print("  Hochrechnung Stufe B (gleiches N_JE=%d, STAPEL=%d):"%(N_JE,STAPEL))
print("  %-12s %8s %10s %8s"%("Schrift","Zeichen","Bigramme","Minuten"))
gesamt=0.0
for nm,z in SCHRIFTEN_PLAN.items():
    n_auf=N_JE*math.ceil(z*z/STAPEL)
    mn=n_auf*s_aufruf/60.0; gesamt+=mn
    DURCHSATZ["hochrechnung_stufeB"][nm]=dict(zeichen=z,bigramme=z*z,minuten=mn)
    print("  %-12s %8d %10d %8.1f"%(nm,z,z*z,mn))
print("  %-12s %8s %10s %8.1f  (alle sieben Schriften)"%("SUMME","","",gesamt))
print("  Anmerkung: faellt Stufe A BUCHSTABEN-ADDITIV aus, reichen in Stufe B")
print("  Stichproben je Schrift statt aller Zellen - siehe LIES_MICH.")
# ---------------- 4  Ablage -------------------------------------------------
BIGRAMM_RESULTS=dict(
    verdict={n:ERG[n]["urteil"] for n in ERG},
    verdict_position={n:ERG[n]["urteil_position"] for n in ERG},
    arch_ok=True,modell=str(globals().get("MODEL_ID","Qwen/Qwen3.6-35B-A3B-FP8")),
    dtype=DTYPE,mappe_seed=MAPPE_SEED,wiederholung=WIEDERHOLUNG,
    alphabet=ALPHABET_NAME,buchstaben=BUCHSTABEN,
    experten=[[s,e] for s,e in EXPERTEN],
    n_je=N_JE,stapel=STAPEL,n_teilungen=N_TEILUNGEN,
    max_new=MAX_NEW,temp=TEMP,n_perm=N_PERM_Z,stopp_ids=STOPP_IDS,
    frame=FRAME,
    tore=dict(min_gueltig=MIN_GUELTIG,min_zellen=MIN_ZELLEN,min_feuer=MIN_FEUER),
    gueltig_anteil=GUELTIG_ANTEIL,korrekt_anteil=KORREKT_ANTEIL,
    kandidaten=dict(primaer=["".join(p) for p in KANDIDATEN_PRIMAER],
                    sekundaer=["".join(p) for p in KANDIDATEN_SEKUNDAER]),
    ergebnis={n:{k:v for k,v in ERG[n].items()} for n in ERG},
    durchsatz=DURCHSATZ,dateien=DATEIEN)
ANTWORTEN_RESULTS=ANTWORTEN
wc_save_all()
print("")
print("FERTIG. Bitte den kompletten Lauf-Ordner zurueckgeben (Drive):")
print("  %s"%RUN_OUT)
print("Darin: BIGRAMM_RESULTS.json (Urteile, Matrizen, p-Werte, Durchsatz),")
print("       ANTWORTEN_RESULTS.json (alle Antworttexte),")
print("       feuerwuerfel_L<Schicht>.npz (Logits/top-8/Token je Reihe,")
print("       alle 256 Experten - die Netzwerk-Suche braucht keinen neuen Lauf),")
print("       protokoll.txt / protokoll_kopie.txt (dieses Protokoll).")


## Rückgabe

Die Zelle schreibt einen Lauf-Ordner nach `Drive/WeirdChat_Runs/bigramm_stufeA_l33_e228_<Zeitstempel>/` und nennt ihn am Ende im Klartext. Bitte diesen Ordner **komplett** zurückgeben, er enthält:

- `BIGRAMM_RESULTS.json` — Urteile je Experte, 26×26-Raten- und Interaktionsmatrix, p-Werte beider Stufen, Positionsbefund, Spitzenzellen, **Durchsatz mit Stufe-B-Hochrechnung**
- `ANTWORTEN_RESULTS.json` — alle Antworttexte mit Gültigkeits-/Korrektheitsflag
- `feuerwuerfel_L<Schicht>.npz` — Routerlogits (float16, alle 256 Experten), top-8, Token und Reihen-Zuordnung je Decode-Position — **die Netzwerk-Suche über weitere Experten braucht keinen neuen GPU-Lauf, sie liest diese Datei**
- `protokoll.txt` / `protokoll_kopie.txt` — das vollständige Protokoll

Bitte außerdem die Durchsatzzahlen aus Abschnitt 3 zurückmelden — an ihnen wird Stufe B (sieben Schriften) dimensioniert.